# LoRA Training on Colab

Run the LoRA DPO training loop on a free Colab T4 GPU.

**Before starting:** Make sure you've selected a GPU runtime: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Verify GPU is available
!nvidia-smi

## 1. Upload repo and install dependencies

From your local machine, create the zip then upload it:
```bash
cd ~/workspace/LoRA && git archive --format=zip HEAD -o /tmp/LoRA.zip
```

In [ ]:
!pip install uv

import os
from google.colab import files

print("Upload LoRA.zip")
uploaded = files.upload()

!mkdir -p /content/LoRA
!unzip -o LoRA.zip -d /content/LoRA
%cd /content/LoRA
!uv sync

## 2. Download base model and create directories

In [ ]:
!bash scripts/00_setup_env.sh

## 3. Upload task data

The `data/` directory is gitignored. Upload your task files here.

In [ ]:
import os
from google.colab import files

os.makedirs("data/tasks", exist_ok=True)

print("Upload train.jsonl and eval.jsonl")
uploaded = files.upload()

for name, content in uploaded.items():
    dest = os.path.join("data/tasks", name)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"Saved {dest} ({len(content)} bytes)")

## 4. Smoke test (1 iteration, small config)

Run this first to verify everything works on the T4. Uses `fp16` (native T4 speed) instead of `bf16`.

In [ ]:
!uv run python scripts/09_run_loop.py --config configs/smoke_test_colab.yaml --run_name colab_smoke

## 5. Full training run (optional)

Once the smoke test passes, run the full loop. Adjust iterations in the config if needed — free Colab sessions can disconnect after ~12h.

In [ ]:
!uv run python scripts/09_run_loop.py --config configs/loop_colab.yaml --run_name colab_full

## 6. Save results to Google Drive (optional)

Mount Drive and copy outputs so they survive session disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DEST = "/content/drive/MyDrive/LoRA_results"

!mkdir -p "{DRIVE_DEST}"
!cp -r models/lora "{DRIVE_DEST}/lora"
!cp -r data/progress "{DRIVE_DEST}/progress"
!cp -r data/rollouts "{DRIVE_DEST}/rollouts"
print(f"Results saved to {DRIVE_DEST}")